# Day 074 — Exercise 4: run_ffmpeg

**What you'll build:** `run_ffmpeg(args, ffmpeg_fn=None) -> dict` — shell out to FFmpeg with injection support for testing.

**Why it matters:** FFmpeg handles every media format conversion, audio extraction, and filter operation that OpenCV cannot. `run_ffmpeg` is the bridge from Python to the full FFmpeg ecosystem.

In [ ]:
from pathlib import Path
_mock_ffmpeg_fn = lambda args: {'returncode': 0, 'stdout': '', 'stderr': ''}

## Task

- **Mock:** `if ffmpeg_fn is not None: return ffmpeg_fn(args)`
- **Real:** `import subprocess; result = subprocess.run(['ffmpeg', '-y'] + list(args), capture_output=True, text=True)`
- Return `{'returncode': result.returncode, 'stdout': result.stdout, 'stderr': result.stderr}`

## Your Implementation

In [ ]:
def run_ffmpeg(args: list, ffmpeg_fn=None) -> dict:
    """Run an FFmpeg command and return the result dict.

    Args:
        args:      FFmpeg arguments (everything after 'ffmpeg -y')
        ffmpeg_fn: callable(args) -> dict for testing
    Returns:
        {'returncode': int, 'stdout': str, 'stderr': str}
    """
    raise NotImplementedError


In [ ]:
def run_ffmpeg(args, ffmpeg_fn=None):
    if ffmpeg_fn is not None:
        return ffmpeg_fn(args)
    import subprocess
    result = subprocess.run(
        ['ffmpeg', '-y'] + list(args),
        capture_output=True, text=True,
    )
    return {
        'returncode': result.returncode,
        'stdout':     result.stdout,
        'stderr':     result.stderr,
    }


## Automated checks

In [ ]:

score, total = 0, 5
try:
    # returns a dict
    result = run_ffmpeg(['-i', 'input.mp4', 'output.avi'], ffmpeg_fn=_mock_ffmpeg_fn)
    assert isinstance(result, dict)
    score += 1; print("✅ returns a dict")

    # required keys present
    for k in ('returncode', 'stdout', 'stderr'):
        assert k in result, f"missing key '{k}'"
    score += 1; print("✅ all required keys present (returncode, stdout, stderr)")

    # returncode is int
    assert isinstance(result['returncode'], int)
    score += 1; print("✅ returncode is int")

    # ffmpeg_fn receives args
    captured = {}
    def _cap(args): captured['args'] = list(args); return {'returncode': 0, 'stdout': '', 'stderr': ''}
    run_ffmpeg(['-i', 'a.mp4', '-vn', 'b.mp3'], ffmpeg_fn=_cap)
    assert captured.get('args') == ['-i', 'a.mp4', '-vn', 'b.mp3']
    score += 1; print("✅ args forwarded to ffmpeg_fn unchanged")

    # mock returns success code 0
    r2 = run_ffmpeg(['any', 'args'], ffmpeg_fn=_mock_ffmpeg_fn)
    assert r2['returncode'] == 0
    score += 1; print("✅ mock ffmpeg_fn returns returncode 0")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def run_ffmpeg(args, ffmpeg_fn=None):
    if ffmpeg_fn is not None:
        return ffmpeg_fn(args)
    import subprocess
    result = subprocess.run(
        ['ffmpeg', '-y'] + list(args),
        capture_output=True, text=True,
    )
    return {
        'returncode': result.returncode,
        'stdout':     result.stdout,
        'stderr':     result.stderr,
    }
```

**Why `-y` before the args?** The `-y` flag must come before the input/output arguments. Prepending it in the base command ensures callers never need to remember to include it. FFmpeg will overwrite existing output files without prompting — essential for automation.

</details>